# 05_4 — Live Scoring con CatBoost residual

Scoring de mercados activos usando `catboost_residual`.

Este notebook **no reentrena**. Lee los artefactos guardados en `data/models/catboost_residual/` y el registro `data/models/registry/` ya actualizado.

In [1]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.config import load_config

cfg = load_config(str(ROOT / 'config' / 'config.yaml'))
REGISTRY = ROOT / 'data' / 'models' / 'registry'
MODEL_DIR = ROOT / 'data' / 'models' / 'catboost_residual'
MODEL_NAME = 'catboost_residual'

print('Registry:', REGISTRY)

Registry: /Users/andres/Documents/ITAM/octavo_semestre/mineria_y_analisis/proyecto_polymarket/data/models/registry


## 1. Cargar modelo y métricas offline

In [2]:
with open(MODEL_DIR / 'run_config.json') as f:
    run_config = json.load(f)
with open(MODEL_DIR / 'test_metrics.json') as f:
    test_metrics = json.load(f)
with open(REGISTRY / 'live_scoring_summary.json') as f:
    live_summary = json.load(f)

tm = test_metrics['test']
raw = tm['raw_metrics']
cal = tm['calibrated_metrics']
mkt = tm['market_baseline_metrics']
beaten = test_metrics.get('market_baseline_beaten', {})
ev = tm['ev_metrics']

print('=== Resumen offline (test set) ===')
print(f"  Brier:    {cal['brier']:.5f}  vs mercado {mkt['brier']:.5f}  {'✓ GANA' if beaten.get('brier') else '✗'}")
print(f"  Log-loss: {cal['log_loss']:.5f}  vs mercado {mkt['log_loss']:.5f}  {'✓ GANA' if beaten.get('log_loss') else '✗'}")
print(f"  ROC-AUC:  {cal['roc_auc']:.5f}")
print()
print(f"  Top-{ev.get('top_k', 'K')} hit rate : {ev.get('top_k_hit_rate', 'N/A')}")
print(f"  Top-{ev.get('top_k', 'K')} avg PnL : {ev.get('top_k_avg_realized_pnl', 'N/A'):.4f}")

=== Resumen offline (test set) ===
  Brier:    0.00421  vs mercado 0.00618  ✓ GANA
  Log-loss: 0.01431  vs mercado 0.02210  ✓ GANA
  ROC-AUC:  0.99993

  Top-100 hit rate : 0.99
  Top-100 avg PnL : 0.1129


## 2. Cargar scores de mercados activos

In [3]:
df_all = pd.read_csv(REGISTRY / 'live_scores.csv')
df_scores = df_all[df_all['model_name'] == MODEL_NAME].copy()

print(f'Mercados activos scored: {len(df_scores):,}')
print(f'Columnas: {list(df_scores.columns)}')
print()

cols_display = ['question', 'price_yes', 'p_yes_calibrated', 'ev_per_share', 'expected_roi_capped', 'days_to_end', 'signal']
display(df_scores[cols_display].head(10))

Mercados activos scored: 529
Columnas: ['model_name', 'id', 'question', 'slug', 'price_yes', 'p_yes_raw', 'p_yes_calibrated', 'ev_per_share', 'expected_roi', 'days_to_end', 'liquidity', 'volume_24h', 'spread', 'end_date', 'signal', 'expected_roi_capped', 'signal_strength']



,question,price_yes,p_yes_calibrated,ev_per_share,expected_roi_capped,days_to_end,signal
1587,"Will Bitcoin reach $70,000 in April?",0.905,1.000,0.095,0.104972,17.327231,HOLD
1588,Will Anthropic have the best AI model at the e...,0.905,1.000,0.095,0.104972,16.160564,HOLD
1589,Will the Virginia redistricting referendum pass?,0.915,1.000,0.085,0.092896,7.160564,HOLD
1590,Will Israel take military action in Gaza on Ap...,0.934,1.000,0.066,0.070664,16.160564,HOLD
1591,Will Shai Gilgeous-Alexander win the 2025–2026...,0.935,1.000,0.065,0.069519,57.160564,HOLD
1592,Will Mojtaba Khamenei be head of state in Iran...,0.567,0.625,0.058,0.102293,261.160564,HOLD
1593,Will Bitcoin hit $60k or $80k first?,0.570,0.625,0.055,0.096491,262.368897,HOLD
1594,Los Angeles Dodgers vs. Toronto Blue Jays,0.575,0.625,0.050,0.086957,0.123758,HOLD
1595,Will the U.S. invade Iran before 2027?,0.585,0.625,0.040,0.068376,261.160564,HOLD
1596,Iran x Israel/US conflict ends by June 30?,0.585,0.625,0.040,0.068376,77.160564,HOLD


## 3. Distribución de señales

In [4]:
signal_counts = df_scores['signal'].value_counts().reindex(['STRONG BUY', 'BUY', 'HOLD'], fill_value=0)
print('Distribución de señales:')
for sig, cnt in signal_counts.items():
    pct = 100 * cnt / len(df_scores)
    print(f'  {sig:12s}: {cnt:4d}  ({pct:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
colors_sig = {'STRONG BUY': '#0D47A1', 'BUY': '#42A5F5', 'HOLD': '#B0BEC5'}
ax.bar(signal_counts.index, signal_counts.values, color=[colors_sig[s] for s in signal_counts.index], edgecolor='white')
for bar, (sig, cnt) in zip(ax.patches, signal_counts.items()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, str(cnt), ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Señales generadas — CatBoost', fontsize=11)
ax.set_ylabel('Mercados')

ax = axes[1]
ax.hist(df_scores['ev_per_share'], bins=50, color='#0D47A1', alpha=0.7, edgecolor='white')
ax.axvline(0, color='red', lw=1.5, ls='--', label='EV = 0')
ax.axvline(0.03, color='#F9A825', lw=1.5, ls='--', label='Umbral BUY (0.03)')
ax.set_xlabel('EV por share')
ax.set_ylabel('Mercados')
ax.set_title('Distribución de EV — mercados activos')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(ROOT / 'figures' / 'catboost_live_signals.png', dpi=150, bbox_inches='tight')
plt.show()

Distribución de señales:
  STRONG BUY  :    0  (0.0%)
  BUY         :    0  (0.0%)
  HOLD        :  529  (100.0%)


/var/folders/dv/l82lzhjj64v3xgj4xs_hdqn80000gn/T/ipykernel_26085/1817691575.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Top mercados por EV y señales accionables

In [5]:
scoring_cfg = cfg.get('scoring', {})
min_liq = scoring_cfg.get('min_liquidity', 1000)
min_vol = scoring_cfg.get('min_volume_24h', 100)
max_spread = scoring_cfg.get('max_spread', 0.10)

df_actionable = df_scores[df_scores['signal'].isin(['BUY', 'STRONG BUY'])].copy()
if 'liquidity' in df_actionable.columns:
    df_actionable = df_actionable[df_actionable['liquidity'] >= min_liq]
if 'volume_24h' in df_actionable.columns:
    df_actionable = df_actionable[df_actionable['volume_24h'] >= min_vol]
if 'spread' in df_actionable.columns:
    df_actionable = df_actionable[df_actionable['spread'] <= max_spread]

df_actionable = df_actionable.sort_values('ev_per_share', ascending=False)
print(f'Señales accionables (post-filtro): {len(df_actionable):,}')
print(f'Filtros: liquidez≥{min_liq}, volumen_24h≥{min_vol}, spread≤{max_spread}')
print()

cols_show = ['question', 'price_yes', 'p_yes_calibrated', 'ev_per_share', 'expected_roi_capped', 'days_to_end', 'signal']
if df_actionable.empty:
    print('No hay BUY/STRONG BUY con los thresholds actuales. Mostrando el top-15 por EV aunque queden en HOLD.')
    fallback = df_scores.sort_values('ev_per_share', ascending=False).head(15)
    display(fallback[cols_show].style.format({
        'price_yes': '{:.3f}',
        'p_yes_calibrated': '{:.3f}',
        'ev_per_share': '{:.4f}',
        'expected_roi_capped': '{:.3f}',
        'days_to_end': '{:.0f}',
    }))
else:
    display(df_actionable[cols_show].head(15).style.format({
        'price_yes': '{:.3f}',
        'p_yes_calibrated': '{:.3f}',
        'ev_per_share': '{:.4f}',
        'expected_roi_capped': '{:.3f}',
        'days_to_end': '{:.0f}',
    }))

Señales accionables (post-filtro): 0
Filtros: liquidez≥1000.0, volumen_24h≥100.0, spread≤0.1

No hay BUY/STRONG BUY con los thresholds actuales. Mostrando el top-15 por EV aunque queden en HOLD.


,question,price_yes,p_yes_calibrated,ev_per_share,expected_roi_capped,days_to_end,signal
1587,"Will Bitcoin reach $70,000 in April?",0.905,1.000,0.0950,0.105,17,HOLD
1588,Will Anthropic have the best AI model at the end of April 2026?,0.905,1.000,0.0950,0.105,16,HOLD
1589,Will the Virginia redistricting referendum pass?,0.915,1.000,0.0850,0.093,7,HOLD
1590,"Will Israel take military action in Gaza on April 5, 2026?",0.934,1.000,0.0660,0.071,16,HOLD
1591,Will Shai Gilgeous-Alexander win the 2025–2026 NBA MVP?,0.935,1.000,0.0650,0.070,57,HOLD
1592,Will Mojtaba Khamenei be head of state in Iran end of 2026?,0.567,0.625,0.0580,0.102,261,HOLD
1593,Will Bitcoin hit $60k or $80k first?,0.570,0.625,0.0550,0.096,262,HOLD
1594,Los Angeles Dodgers vs. Toronto Blue Jays,0.575,0.625,0.0500,0.087,0,HOLD
1596,Iran x Israel/US conflict ends by June 30?,0.585,0.625,0.0400,0.068,77,HOLD
1595,Will the U.S. invade Iran before 2027?,0.585,0.625,0.0400,0.068,261,HOLD


## 5. Backtest offline en test set — por horizonte

In [6]:
by_horizon = tm.get('by_horizon', {})

bucket_rows = []
for bucket_name, bucket_data in by_horizon.items():
    prob_m = bucket_data.get('probability_metrics', {})
    mkt_m = bucket_data.get('market_baseline_metrics', {})
    ev_m = bucket_data.get('ev_metrics', {})
    bucket_rows.append({
        'Horizonte': bucket_name.replace('_', ' '),
        'n': bucket_data.get('count', 0),
        'Brier CatBoost': prob_m.get('brier'),
        'Brier Mkt': mkt_m.get('brier'),
        'ROC-AUC': prob_m.get('roc_auc'),
        'Top-K hit': ev_m.get('top_k_hit_rate'),
        'Top-K PnL': ev_m.get('top_k_avg_realized_pnl'),
    })

df_bkt = pd.DataFrame(bucket_rows)
display(df_bkt.style.format({
    'Brier CatBoost': '{:.4f}',
    'Brier Mkt': '{:.4f}',
    'ROC-AUC': '{:.4f}',
    'Top-K hit': '{:.0%}',
    'Top-K PnL': '{:.4f}',
}))

,Horizonte,n,Brier CatBoost,Brier Mkt,ROC-AUC,Top-K hit,Top-K PnL
0,short 1 3d,2380,0.0032,0.0052,1.0000,100%,0.0255
1,medium 4 14d,2380,0.0032,0.0052,1.0000,100%,0.0264
2,long 15plus,1167,0.0085,0.0101,0.9997,99%,0.0692


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
x = np.arange(len(df_bkt))
w = 0.35
ax.bar(x - w/2, df_bkt['Brier CatBoost'], w, label='CatBoost', color='#0D47A1')
ax.bar(x + w/2, df_bkt['Brier Mkt'], w, label='Mercado', color='#B0BEC5')
ax.set_xticks(x)
ax.set_xticklabels(df_bkt['Horizonte'], rotation=10)
ax.set_title('Brier por horizonte (test)')
ax.legend()

ax = axes[1]
pnl_vals = df_bkt['Top-K PnL'].fillna(0)
hit_vals = df_bkt['Top-K hit'].fillna(0)
colors_b = ['#0D47A1' if v > 0 else '#EF9A9A' for v in pnl_vals]
bars = ax.bar(df_bkt['Horizonte'], pnl_vals, color=colors_b, edgecolor='white')
ax2 = ax.twinx()
ax2.plot(df_bkt['Horizonte'], hit_vals * 100, 'o--', color='#F9A825', lw=2, label='Hit rate %')
ax2.set_ylabel('Hit rate (%)', color='#F9A825')
ax2.tick_params(axis='y', colors='#F9A825')
ax.axhline(0, color='black', lw=1, ls='--')
ax.set_title('Top-K PnL y Hit Rate por horizonte (test)')
ax.set_ylabel('PnL promedio')
for bar, v in zip(bars, pnl_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003, f'{v:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(ROOT / 'figures' / 'catboost_live_backtest.png', dpi=150, bbox_inches='tight')
plt.show()

/var/folders/dv/l82lzhjj64v3xgj4xs_hdqn80000gn/T/ipykernel_26085/1499764236.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Resumen operativo

In [8]:
n_buy = (df_scores['signal'] == 'BUY').sum()
n_sbuy = (df_scores['signal'] == 'STRONG BUY').sum()
n_action = len(df_actionable)

print('Lectura rápida:')
print(f"  Mercados activos analizados  : {len(df_scores):,}")
print(f"  Señales BUY                  : {n_buy}")
print(f"  Señales STRONG BUY           : {n_sbuy}")
print(f"  Accionables (post-liquidez)  : {n_action}")
print()
print('Validación offline (test set):')
print(f"  Brier    : {cal['brier']:.5f}  vs mercado {mkt['brier']:.5f}  {'✓ GANA' if beaten.get('brier') else '✗ pierde'}")
print(f"  Log-loss : {cal['log_loss']:.5f}  vs mercado {mkt['log_loss']:.5f}  {'✓ GANA' if beaten.get('log_loss') else '✗ pierde'}")
print(f"  Top-K hit rate : {ev.get('top_k_hit_rate')}")
print(f"  Top-K PnL prom : {ev.get('top_k_avg_realized_pnl', 0):.4f}")
print()
if n_action == 0:
    print('Lectura live: modelo muy conservador con los thresholds actuales; hoy funciona mejor como filtro probabilístico offline que como motor de señales.')
else:
    print('Lectura live: el modelo sí está encontrando oportunidades accionables en el snapshot actual.')
print()
print('Para regenerar el scoring:')
print('  python -m src.scoring.scorer --config config/config.yaml --all-models')

Lectura rápida:
  Mercados activos analizados  : 529
  Señales BUY                  : 0
  Señales STRONG BUY           : 0
  Accionables (post-liquidez)  : 0

Validación offline (test set):
  Brier    : 0.00421  vs mercado 0.00618  ✓ GANA
  Log-loss : 0.01431  vs mercado 0.02210  ✓ GANA
  Top-K hit rate : 0.99
  Top-K PnL prom : 0.1129

Lectura live: modelo muy conservador con los thresholds actuales; hoy funciona mejor como filtro probabilístico offline que como motor de señales.

Para regenerar el scoring:
  python -m src.scoring.scorer --config config/config.yaml --all-models
